# 05 · A Log-Triage Agent — with a Gradio UI

**Where we are in the stack:** the **Agent = control plane** again - the same
loop from notebook 2. Only the resource changed: this agent reads **log files**
and answers operational questions about them ("what's failing, and where?").

Same loop:

```
prompt the model  ->  did it ask for a tool?
      ^                       |  yes                 |  no
      |                  call the tool          FINAL ANSWER -> stop
      |                       |
      +----- feed result back +
```

Input: a **logs path** + a **question**. The agent must `ls`, `grep`, `tail`
and *count* its way to an answer - it cannot invent log lines.

> Tools are **read-only** and **sandboxed** to a logs root. Look, don't touch.

## 0. Provider config (loads `.env`)

Needs a **tool-capable** model. Settings come from `.env` (`OPENAI_BASE_URL`, `OPENAI_API_KEY`, `MODEL`).

In [ ]:
# --- Provider config: loads .env, works with any OpenAI-compatible server ---
import os
from openai import OpenAI

# Load .env if python-dotenv is available; otherwise parse it by hand.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    if os.path.exists(".env"):
        for line in open(".env"):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. A sample logs directory (so the demo is self-contained)

We generate a couple of fake log files with a deterministic mix of INFO / WARN /
ERROR lines. Point `ROOT` at real logs instead and nothing else changes.

In [ ]:
import os, random

ROOT = os.path.abspath("sample_logs")
os.makedirs(ROOT, exist_ok=True)

random.seed(7)
levels = ["INFO"]*7 + ["WARN"]*2 + ["ERROR"]
msgs = {
    "INFO":  ["request handled", "cache hit", "user login", "job complete"],
    "WARN":  ["high latency", "retry scheduled", "cache miss storm"],
    "ERROR": ["connection refused", "timeout to db", "null pointer", "disk full"],
}
for fname, n in [("api.log", 60), ("worker.log", 40)]:
    with open(os.path.join(ROOT, fname), "w") as f:
        for i in range(n):
            lvl = random.choice(levels)
            f.write(f"2026-06-23T04:{i%60:02d}:00 {lvl} {fname[:-4]}: {random.choice(msgs[lvl])}\n")
print("created logs in", ROOT)
print(open(os.path.join(ROOT, "api.log")).read().splitlines()[:3])

## 2. The tools = a tiny read-only log shell

Four functions, each a familiar move when triaging logs. All paths resolve under
`ROOT`; nothing escapes, nothing writes.

In [ ]:
import subprocess

def _safe(path):
    full = os.path.abspath(os.path.join(ROOT, path))
    if full != ROOT and not full.startswith(ROOT + os.sep):
        raise ValueError(f"path '{path}' escapes the logs root")
    return full

def list_logs(path="."):
    """ls: list log files at `path` (relative to the logs root)."""
    full = _safe(path)
    if not os.path.isdir(full):
        return {"error": f"not a directory: {path}"}
    return {"path": path, "entries": sorted(os.listdir(full))}

def tail(path, n=15):
    """tail -n: last n lines of a log file."""
    full = _safe(path)
    if not os.path.isfile(full):
        return {"error": f"not a file: {path}"}
    lines = open(full, errors="replace").read().splitlines()
    return {"path": path, "lines": lines[-n:]}

def grep_logs(pattern, path=".", max_matches=40):
    """grep -rn: search log contents for a pattern (e.g. 'ERROR')."""
    full = _safe(path)
    out = subprocess.run(["grep", "-rIn", "--", pattern, full],
                         capture_output=True, text=True).stdout
    matches = [l.replace(ROOT + os.sep, "") for l in out.splitlines()[:max_matches]]
    return {"pattern": pattern, "matches": matches, "count": len(matches)}

def count_pattern(pattern, path="."):
    """grep -rc: total number of lines matching `pattern` under `path`."""
    full = _safe(path)
    out = subprocess.run(["grep", "-rIc", "--", pattern, full],
                         capture_output=True, text=True).stdout
    total = sum(int(l.rsplit(":", 1)[-1]) for l in out.splitlines() if l.strip())
    return {"pattern": pattern, "total": total}

# Prove the tools work without the model
print(list_logs("."))
print(count_pattern("ERROR"))

## 3. Describe the tools to the model (JSON schema)

The model only *requests* a call; **you** execute it and feed the result back -
the capability-table pattern from notebook 2.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "list_logs",
        "description": "List log files in the logs directory. Like `ls`.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string", "description": "dir, default '.'"}},
            "required": []}}},
    {"type": "function", "function": {
        "name": "tail",
        "description": "Return the last n lines of a log file. Like `tail -n`.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string"}, "n": {"type": "integer", "description": "lines, default 15"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "grep_logs",
        "description": "Search log contents for a pattern, return file:line:text. Like `grep -rn`.",
        "parameters": {"type": "object",
            "properties": {"pattern": {"type": "string"}, "path": {"type": "string", "description": "dir, default '.'"}},
            "required": ["pattern"]}}},
    {"type": "function", "function": {
        "name": "count_pattern",
        "description": "Count how many lines match a pattern across the logs. Like `grep -c`.",
        "parameters": {"type": "object",
            "properties": {"pattern": {"type": "string"}, "path": {"type": "string", "description": "dir, default '.'"}},
            "required": ["pattern"]}}},
]

TOOL_REGISTRY = {"list_logs": list_logs, "tail": tail,
                 "grep_logs": grep_logs, "count_pattern": count_pattern}

## 4. The agent loop (the same FSM as notebook 2)

Identical control plane; only the system prompt and tools differ.

In [ ]:
import json

SYSTEM = """You are a log-triage assistant.
Answer ONLY from the log files, using your tools.
Strategy: list the files, count error/warn levels, grep for the failing patterns,
then tail the relevant file to see context. Cite file names and line numbers.
Be concise. Never invent log lines."""

def triage(question, max_iterations=12, temperature=0):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Logs root: {ROOT}\n\nQuestion: {question}"},
    ]
    print("ROOT:", ROOT); print("Q:", question); print("=" * 72)
    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=temperature)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            print(f"\n[step {step}] FINAL ANSWER\n{'-'*72}\n{msg.content}")
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            name = tc.function.name; args = json.loads(tc.function.arguments)
            print(f"[step {step}] TOOL  -> {name}({args})")
            try: result = TOOL_REGISTRY[name](**args)
            except Exception as e: result = {"error": str(e)}
            preview = str(result)
            print(f"[step {step}] RESULT <- {preview[:200]}{'...' if len(preview) > 200 else ''}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)[:4000]})
    return "Stopped: hit max_iterations (TTL expired)."

## 5. Run it - triage the logs

In [ ]:
_ = triage("Which log has the most errors, what are the top error types, and show me a few example lines.")

## 6. Run it - a targeted question

In [ ]:
_ = triage("Are there any database-related failures? If so, which file and how many?")

## Recap

Third agent, **same loop** - we just pointed the tools at logs instead of code or
a network. An agent is a control-plane FSM plus a capability table; swap the
table and the behaviour changes, the loop does not.

Production triage tools add retries, streaming, persistent memory and
observability (notebook 2's "what's missing" list). We stayed **read-only** here.

**Next:** point the same idea at a *database* with read-only SQL. ->
`06_sql_data_agent.ipynb`

## 7. Interactive UI (Gradio)

Ask the agent about your logs. Point it at any logs directory; the **Agent trace**
box shows it `grep`/`tail`/`count` its way to a verdict.

In [ ]:
# Install Gradio for the interactive UI (run once).
%pip install -q gradio

In [ ]:
import io, contextlib

def _run_capture(fn, *args, **kwargs):
    """Run an agent function, capturing its printed step-by-step trace."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            result = fn(*args, **kwargs)
        except Exception as e:
            result = f"Error: {e}"
    return buf.getvalue(), (result or "")

In [ ]:
import gradio as gr

def triage_ui(logs_path, question):
    global ROOT
    ROOT = os.path.abspath(logs_path or "sample_logs")
    trace, answer = _run_capture(triage, question)
    return trace, answer

with gr.Blocks(title="05 · Log Triage") as demo:
    gr.Markdown("# 🚨 Log-Triage Agent\n"
                "Read-only, sandboxed log tools (ls / tail / grep / count). "
                "Ask what is failing; it greps the logs to find out.")
    logs_path = gr.Textbox(label="Logs directory", value="sample_logs")
    question = gr.Textbox(
        label="Question",
        value="Which log has the most errors, what are the top error types, and show me a few example lines.",
    )
    run = gr.Button("Triage", variant="primary")
    gr.Examples(
        ["Which log has the most errors, what are the top error types, and show me a few example lines.",
         "Are there any database-related failures? If so, which file and how many?"],
        inputs=question,
    )
    answer = gr.Markdown(label="Triage verdict")
    trace = gr.Textbox(label="Agent trace (tool calls)", lines=16)
    run.click(triage_ui, inputs=[logs_path, question], outputs=[trace, answer])

demo.launch()